In [ ]:
# DDGR20，嵌入近似hint，l-noisy < <v,s> < l+noisy
cd framework

In [ ]:
load("../framework/instance_gen.sage")
import numpy as np
import random

In [ ]:
n = 1024
m = n
q = 3329
D_s = build_centered_binomial_law(2)
D_e = D_s
A, b, s, dbdd = initialize_from_LWE_instance(DBDD_predict, n, q, m, D_e, D_s)
_ = dbdd.integrate_q_vectors(q, report_every=20)
beta, delta = dbdd.estimate_attack()

In [ ]:
def generate_se_eta_bias_approx_hint(m, n, bias, k):
    V = []
    L = []

    for i in range(k):
        D_e = {-3: 1/64, -2: 6/64, -1: 15/64, 0: 20/64, 1: 15/64, 2: 6/64, 3: 1/64}
        values, probabilities = zip(*D_e.items())
        v = np.array(np.random.choice(values, size=m+n, p=probabilities))
        noisy = random.randint(-bias, bias)
        l = dbdd.leak(v)+noisy
        V.append(v)
        L.append(l)
    print("L",L)
    return V,L

In [ ]:
num_hint = 1000
bias = int(q/64)
V, L = generate_se_eta_bias_approx_hint(m, n, bias, num_hint)

In [ ]:
for j in range(num_hint):
    print("the ",j+1,"-th secret error approx hint")
    _ = dbdd.integrate_approx_hint(vec(V[j]), L[j], bias, aposteriori=False)
    _ = dbdd.integrate_q_vectors(q, report_every=20)


In [ ]:
nph_Kyber1024_sca = [0, 40, 80, 120, 160, 200, 240, 280, 320, 360, 400, 440, 480, 520, 560, 600, 640, 680, 720, 760, 800, 840, 880, 920, 960, 1000]
BETA = []
index = 0

for j in range(num_hint+1):
    if j == nph_Kyber1024_sca[index]:
        _ = dbdd.integrate_q_vectors(q, report_every=20)
        beta, delta = dbdd.estimate_attack()
        print("beta: ", beta)
        BETA.append(beta)
        index += 1
    print("the ",j+1,"-th secret error approx hint")
    _ = dbdd.integrate_approx_hint(vec(V[j]), L[j], bias, aposteriori=False)